# Analyse d'erreurs et explicabilité d'un classifieur d'images

Un classifieur de photos de paysages (6 classes : bâtiments, forêt, glacier, montagne, mer, rue) se trompe sur environ une image sur dix. Sur quelles images, pourquoi, et faut-il changer de modèle ou corriger les données ? Les images sont celles du jeu [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification), des photos de 150 × 150 pixels.

1. Un modèle pré-entraîné
2. L'entraînement de sa dernière couche
3. La matrice de confusion et les paires de classes confondues
4. Expliquer des erreurs avec SHAP
5. Modèle ou étiquettes ?

Le notebook tourne sans GPU, en une dizaine de minutes : le réseau pré-entraîné reste gelé, ses sorties sont calculées une seule fois, sur un sous-ensemble des images d'entraînement.

## Données

Les images sont rangées dans un dossier par classe : `landscape/seg_train` (14 034 images) et `landscape/seg_test` (3 000). `get_loader` les charge avec les transformations du modèle pré-entraîné, `WEIGHTS.transforms()` (redimensionnement en 224 × 224 et normalisation d'ImageNet). Pour tenir en quelques minutes sans GPU, le modèle ne s'entraîne que sur `TRAIN_PER_CLASS` images par classe, tirées au hasard ; le jeu de test est complet.

In [ ]:
# shap est dans Colab, pas dans le Codespace
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import PIL.Image
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHTS = models.MobileNet_V3_Small_Weights.DEFAULT
TRAIN_PER_CLASS = 300  # images d'entraînement par classe


def get_loader(
  directory: str, shuffle: bool = False, per_class: int | None = None
) -> DataLoader:
  dataset = datasets.ImageFolder(directory, transform=WEIGHTS.transforms())
  if per_class is not None:  # per_class images tirées au hasard dans chaque classe
    random.seed(0)
    dataset.samples = [
      sample
      for target in range(len(dataset.classes))
      for sample in random.sample(
        [s for s in dataset.samples if s[1] == target], per_class
      )
    ]
    dataset.targets = [target for _, target in dataset.samples]
  return DataLoader(dataset, batch_size=64, shuffle=shuffle)


train_image_loader = get_loader("landscape/seg_train", per_class=TRAIN_PER_CLASS)
test_image_loader = get_loader("landscape/seg_test")
CLASSES = test_image_loader.dataset.classes
train_paths = [path for path, _ in train_image_loader.dataset.samples]
test_paths = [path for path, _ in test_image_loader.dataset.samples]
print(f"Classes : {CLASSES}")
print(f"{len(train_paths)} images d'entraînement, {len(test_paths)} de test")

## 1. Un modèle pré-entraîné

Créez `model`, un MobileNetV3 *small* pré-entraîné sur ImageNet (les poids `WEIGHTS`). Gelez sa partie convolutive, `model.features`, et remplacez sa dernière couche, `model.classifier[-1]`, par une couche linéaire à une sortie par classe de `CLASSES`. Seules les couches de `model.classifier` restent entraînables : la première cellule de la partie suivante compte les paramètres.

**Documentation utile**

- [`mobilenet_v3_small`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_small.html)
- [`Module.requires_grad_`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.requires_grad_)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `models.mobilenet_v3_small(weights=WEIGHTS)` : le réseau et ses poids pré-entraînés
- `model.features.requires_grad_(False)` : plus de gradient pour les paramètres de la partie convolutive
- `model.classifier` est un `nn.Sequential` dont la dernière couche, `nn.Linear(1024, 1000)`, sort les 1 000 classes d'ImageNet ; `model.classifier[-1].in_features` vaut 1024
- Pour vérifier : 596 998 paramètres entraînables sur 1 524 006

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
model = models.mobilenet_v3_small(weights=...)
model.features.requires_grad_(...)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, ...)
model = model.to(DEVICE)
```

### Solution

In [ ]:
model = models.mobilenet_v3_small(weights=WEIGHTS)
model.features.requires_grad_(False)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(CLASSES))
model = model.to(DEVICE)

## 2. L'entraînement de la dernière couche

La partie gelée renvoie toujours le même vecteur de 576 caractéristiques pour une image donnée : `extract_features` le calcule une fois pour toutes, pour chaque image (quelques minutes sans GPU). Les couches de `model.classifier` s'entraînent ensuite sur ces vecteurs, en quelques secondes, avec les fonctions habituelles `train_epoch`, `evaluate` et `predict`.

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"{trainable:,} paramètres entraînables sur {total:,}")


@torch.inference_mode()
def extract_features(model: nn.Module, loader: DataLoader) -> TensorDataset:
  """Les sorties de la partie gelée (features puis avgpool), et les étiquettes."""
  model.eval()
  features, labels = [], []
  for images, targets in loader:
    features.append(model.avgpool(model.features(images.to(DEVICE))).flatten(1).cpu())
    labels.append(targets)
  return TensorDataset(torch.cat(features), torch.cat(labels))


train_features = extract_features(model, train_image_loader)
test_features = extract_features(model, test_image_loader)
print(f"Caractéristiques de test : {tuple(test_features.tensors[0].shape)}")

In [ ]:
def train_epoch(model: nn.Module, loader: DataLoader, optimizer) -> float:
  """Une époque ; renvoie la perte moyenne."""
  model.train()
  loss_fn = nn.CrossEntropyLoss()
  total_loss = 0.0
  for x, y in loader:
    x, y = x.to(DEVICE), y.to(DEVICE)
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()
    total_loss += loss.item() * len(x)
  return total_loss / len(loader.dataset)


@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader) -> float:
  """Exactitude (accuracy) sur loader."""
  model.eval()
  correct = 0
  for x, y in loader:
    correct += (model(x.to(DEVICE)).argmax(dim=1).cpu() == y).sum().item()
  return correct / len(loader.dataset)


@torch.inference_mode()
def predict(model: nn.Module, loader: DataLoader) -> torch.Tensor:
  """Indices des classes prédites."""
  model.eval()
  return torch.cat([model(x.to(DEVICE)).argmax(dim=1).cpu() for x, _ in loader])


torch.manual_seed(0)
train_loader = DataLoader(train_features, batch_size=64, shuffle=True)
test_loader = DataLoader(test_features, batch_size=64)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=0.001)
for epoch in range(10):
  loss = train_epoch(model.classifier, train_loader, optimizer)
  print(f"Époque {epoch + 1} : perte {loss:.3f}")
print(f"Exactitude sur le jeu de test : {evaluate(model.classifier, test_loader):.3f}")

## 3. La matrice de confusion et les paires confondues

Calculez `test_preds`, les classes que le modèle prédit pour les images de test, et `test_labels`, leurs vraies classes, en tableaux NumPy, puis affichez la matrice de confusion. Calculez ensuite `pairs`, les trois paires (vraie classe, classe prédite) les plus fréquentes parmi les erreurs, avec leur nombre d'images. Affichez enfin, avec `show_images`, une dizaine d'images de la paire la plus fréquente. Ces erreurs se ressemblent-elles ? Les deux classes de la paire se distinguent-elles toujours à l'œil ?

**Documentation utile**

- [`ConfusionMatrixDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)
- [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [`numpy.flatnonzero`](https://numpy.org/doc/stable/reference/generated/numpy.flatnonzero.html)

In [ ]:
def show_images(paths: list[str], titles: list[str]) -> None:
  """Affiche des images côte à côte, avec leurs titres."""
  _, axes = plt.subplots(1, len(paths), figsize=(2 * len(paths), 2.4), squeeze=False)
  for ax, path, title in zip(axes[0], paths, titles):
    ax.imshow(PIL.Image.open(path))
    ax.set_title(title, fontsize=9)
    ax.axis("off")
  plt.show()


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `test_features.tensors[1]` : les étiquettes du jeu de test, dans l'ordre de `test_paths` ; `predict(model.classifier, test_loader)` : les classes prédites, dans le même ordre ; `.numpy()` les convertit
- `ConfusionMatrixDisplay.from_predictions(test_labels, test_preds, display_labels=CLASSES, xticks_rotation=45)` : les vraies classes en lignes, les prédites en colonnes
- `confusion_matrix(test_labels, test_preds)[true, predicted]` : le nombre d'images de la classe `true` prédites `predicted` ; les erreurs sont hors de la diagonale (`true != predicted`)
- `sorted(..., reverse=True)[:3]` sur des triplets `(nombre, vraie classe, classe prédite)` : les trois plus grands nombres d'abord
- `np.flatnonzero((test_labels == a) & (test_preds == b))` : les positions des images de la classe `a` prédites `b`, qui sont aussi leurs positions dans `test_paths`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

test_labels = test_features.tensors[1].numpy()
test_preds = predict(..., ...).numpy()
ConfusionMatrixDisplay.from_predictions(
  ..., ..., display_labels=CLASSES, xticks_rotation=45
)
plt.show()

matrix = confusion_matrix(test_labels, test_preds)
pairs = sorted(
  (
    (matrix[true, predicted], CLASSES[true], CLASSES[predicted])
    for true in range(len(CLASSES))
    for predicted in range(len(CLASSES))
    if ...
  ),
  reverse=True,
)[:3]
for count, true, predicted in pairs:
  print(f"{count} images « {true} » prédites « {predicted} »")

count, true, predicted = pairs[0]
mistakes = np.flatnonzero(
  (test_labels == CLASSES.index(true)) & (... == CLASSES.index(...))
)
show_images([test_paths[i] for i in mistakes[:10]], [f"n° {i}" for i in mistakes[:10]])
```

### Solution

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

test_labels = test_features.tensors[1].numpy()
test_preds = predict(model.classifier, test_loader).numpy()
ConfusionMatrixDisplay.from_predictions(
  test_labels, test_preds, display_labels=CLASSES, xticks_rotation=45
)
plt.show()

matrix = confusion_matrix(test_labels, test_preds)
pairs = sorted(
  (
    (matrix[true, predicted], CLASSES[true], CLASSES[predicted])
    for true in range(len(CLASSES))
    for predicted in range(len(CLASSES))
    if true != predicted
  ),
  reverse=True,
)[:3]
for count, true, predicted in pairs:
  print(f"{count} images « {true} » prédites « {predicted} »")

count, true, predicted = pairs[0]
mistakes = np.flatnonzero(
  (test_labels == CLASSES.index(true)) & (test_preds == CLASSES.index(predicted))
)
show_images([test_paths[i] for i in mistakes[:10]], [f"n° {i}" for i in mistakes[:10]])

## 4. Expliquer des erreurs avec SHAP

Les montagnes prises pour des glaciers comptent parmi les erreurs les plus fréquentes. Pour voir ce qui pousse le modèle vers « glacier », expliquez-en six avec SHAP, celles de `images`. Écrivez d'abord `predict_proba(images)`, qui prend un lot d'images NumPy de forme (N, 150, 150, 3), aux valeurs entre 0 et 1, et renvoie les probabilités des 6 classes, de forme (N, 6) : SHAP n'appelle le modèle qu'à travers cette fonction. Créez ensuite un explainer qui masque des zones des images en les remplaçant par du gris, calculez `explanation` pour les sorties « glacier » et « mountain », et affichez-la. Quelles zones des images poussent le modèle vers « glacier » ? Vers « mountain » ?

**Documentation utile**

- [`Tensor.permute`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.permute.html)
- [`shap.Explainer`](https://shap.readthedocs.io/en/latest/generated/shap.Explainer.html)
- [`shap.maskers.Image`](https://shap.readthedocs.io/en/latest/generated/shap.maskers.Image.html)
- [`shap.plots.image`](https://shap.readthedocs.io/en/latest/generated/shap.plots.image.html)

In [ ]:
GLACIER, MOUNTAIN = CLASSES.index("glacier"), CLASSES.index("mountain")
mountains_as_glaciers = np.flatnonzero(
  (test_labels == MOUNTAIN) & (test_preds == GLACIER)
)
print(f"{len(mountains_as_glaciers)} montagnes prédites glacier")


def load_image(path: str) -> np.ndarray:
  """Une image en tableau (150, 150, 3), aux valeurs entre 0 et 1."""
  image = PIL.Image.open(path).convert("RGB").resize((150, 150))
  return np.asarray(image, dtype=np.float32) / 255


images = np.stack([load_image(test_paths[i]) for i in mountains_as_glaciers[:6]])


def predict_proba(images: np.ndarray) -> np.ndarray: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.as_tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)` : d'un lot NumPy (N, hauteur, largeur, canaux) à un tenseur PyTorch (N, canaux, hauteur, largeur)
- `WEIGHTS.transforms()(batch)` : le même redimensionnement et la même normalisation que pendant l'entraînement
- `model(...).softmax(dim=1)`, sous `torch.inference_mode()` après `model.eval()`, puis `.cpu().numpy()`
- Pour vérifier : chaque ligne de `predict_proba(images)` a sa plus grande probabilité dans la colonne `GLACIER` (2), et sa somme vaut 1
- `shap.maskers.Image(0.5, images[0].shape)` : les zones masquées sont remplacées par du gris (0,5) ; le masqueur `"blur(32,32)"` des exemples de shap demande OpenCV
- `shap.Explainer(predict_proba, masker, output_names=CLASSES)` : avec un masqueur d'images, shap choisit l'explainer *Partition*, qui découpe l'image en blocs de plus en plus petits
- `explainer(images, max_evals=500, batch_size=50, outputs=[GLACIER, MOUNTAIN])` : 500 appels au modèle par image (une quinzaine de secondes par image sans GPU), par lots de 50 ; `outputs` : les classes à expliquer
- `shap.plots.image(explanation)` : en rouge, les zones qui poussent vers la classe de la colonne ; en bleu, celles qui en éloignent

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import shap


@torch.inference_mode()
def predict_proba(images: np.ndarray) -> np.ndarray:
  model.eval()
  batch = torch.as_tensor(images, dtype=torch.float32).permute(...)
  return model(WEIGHTS.transforms()(batch).to(DEVICE)).softmax(dim=...).cpu().numpy()


print(predict_proba(images).round(2))

masker = shap.maskers.Image(0.5, images[0].shape)
explainer = shap.Explainer(..., masker, output_names=CLASSES)
explanation = explainer(images, max_evals=500, batch_size=50, outputs=[..., ...])
shap.plots.image(...)
```

### Solution

In [ ]:
import shap


@torch.inference_mode()
def predict_proba(images: np.ndarray) -> np.ndarray:
  model.eval()
  batch = torch.as_tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)
  return model(WEIGHTS.transforms()(batch).to(DEVICE)).softmax(dim=1).cpu().numpy()


print(predict_proba(images).round(2))

masker = shap.maskers.Image(0.5, images[0].shape)
explainer = shap.Explainer(predict_proba, masker, output_names=CLASSES)
explanation = explainer(
  images, max_evals=500, batch_size=50, outputs=[GLACIER, MOUNTAIN]
)
shap.plots.image(explanation)

## 5. Modèle ou étiquettes ?

Une erreur vient du modèle quand l'image est claire, mais qu'il s'appuie sur un mauvais indice. Elle vient des étiquettes quand des images semblables portent tantôt une étiquette, tantôt l'autre : aucun modèle ne peut alors apprendre la frontière. Pour trancher, écrivez `neighbours(indices, k)`, qui renvoie, pour chaque image de test de `indices`, les positions dans `train_paths` des `k` images d'entraînement les plus proches, dans l'espace des caractéristiques du réseau gelé (les vecteurs de `test_features` et de `train_features`, comparés par similarité cosinus). Comparez la part d'étiquettes « glacier » parmi les 5 voisines des montagnes prédites glacier, et parmi celles des montagnes bien classées. Affichez enfin les trois premières erreurs de `mountains_as_glaciers` et leurs voisines, avec leurs étiquettes. D'après ces voisines et les figures de SHAP, ces erreurs viennent-elles plutôt du modèle ou des étiquettes ? Que faudrait-il faire ?

**Documentation utile**

- [`torch.nn.functional.normalize`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html)
- [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)

In [ ]:
def neighbours(indices: np.ndarray, k: int = 5) -> np.ndarray: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `normalize(vectors, dim=1)` (de `torch.nn.functional`) : chaque vecteur ramené à une norme de 1 ; le produit scalaire de deux vecteurs normalisés est leur similarité cosinus
- `test_vectors[indices] @ train_vectors.T` : la similarité de chaque image de test de `indices` avec chaque image d'entraînement, de forme (len(indices), 1 800)
- `similarities.topk(k, dim=1).indices` : les positions des `k` plus grandes similarités de chaque ligne ; ce sont des positions dans `train_paths` et dans `train_features`
- `train_labels[neighbours(indices)]` : les étiquettes des voisines, de forme (len(indices), k) ; `(... == GLACIER).mean()` : la part de « glacier »
- Pour vérifier : `neighbours(mountains_as_glaciers[:2]).shape` vaut `(2, 5)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from torch.nn.functional import normalize

train_vectors = normalize(train_features.tensors[0], dim=1)
test_vectors = normalize(..., dim=1)
train_labels = train_features.tensors[1].numpy()


def neighbours(indices: np.ndarray, k: int = 5) -> np.ndarray:
  similarities = test_vectors[indices] @ train_vectors.T
  return similarities.topk(..., dim=1).indices.numpy()


well_classified = np.flatnonzero((test_labels == MOUNTAIN) & (test_preds == ...))
for name, indices in [
  ("Montagnes prédites glacier", mountains_as_glaciers),
  ("Montagnes bien classées", well_classified),
]:
  share = (train_labels[neighbours(indices)] == ...).mean()
  print(f"{name} : {share:.0%} de voisines étiquetées « glacier »")

for i, near in zip(mountains_as_glaciers[:3], neighbours(mountains_as_glaciers[:3])):
  show_images(
    [test_paths[i]] + [train_paths[j] for j in near],
    [f"test n° {i}"] + [CLASSES[...] for j in near],
  )
```

### Solution

In [ ]:
from torch.nn.functional import normalize

train_vectors = normalize(train_features.tensors[0], dim=1)
test_vectors = normalize(test_features.tensors[0], dim=1)
train_labels = train_features.tensors[1].numpy()


def neighbours(indices: np.ndarray, k: int = 5) -> np.ndarray:
  similarities = test_vectors[indices] @ train_vectors.T
  return similarities.topk(k, dim=1).indices.numpy()


well_classified = np.flatnonzero((test_labels == MOUNTAIN) & (test_preds == MOUNTAIN))
for name, indices in [
  ("Montagnes prédites glacier", mountains_as_glaciers),
  ("Montagnes bien classées", well_classified),
]:
  share = (train_labels[neighbours(indices)] == GLACIER).mean()
  print(f"{name} : {share:.0%} de voisines étiquetées « glacier »")

for i, near in zip(mountains_as_glaciers[:3], neighbours(mountains_as_glaciers[:3])):
  show_images(
    [test_paths[i]] + [train_paths[j] for j in near],
    [f"test n° {i}"] + [CLASSES[train_labels[j]] for j in near],
  )

Éléments de réponse :

- Les montagnes prédites glacier ont bien plus de voisines « glacier » que les montagnes bien classées : dans le jeu d'entraînement, des photos très proches portent l'étiquette « glacier ». Le modèle reproduit l'ambiguïté des étiquettes. La paire inverse, les glaciers prédits montagne, est d'ailleurs l'autre erreur la plus fréquente.
- Les figures de SHAP vont dans le même sens : le rouge de « glacier » tombe sur la neige et la glace (sommets enneigés, champs de neige, fronts de glace), des indices sensés pour cette classe, et non sur un détail sans rapport. Plusieurs de ces « montagnes » montrent d'ailleurs un glacier : un humain hésiterait aussi, et certaines étiquettes sont fausses.
- Ces erreurs viennent donc surtout des étiquettes. On les traite dans les données : préciser le guide d'annotation (par exemple « glacier dès que la glace ou la neige occupe l'essentiel de l'image »), ré-annoter les cas ambigus, ou admettre plusieurs étiquettes par image. Un meilleur modèle n'y gagnerait presque rien : sur cette tranche, la performance humaine de référence est elle-même basse.
- L'erreur viendrait du modèle si ses voisines étaient des montagnes et que SHAP pointait un indice sans rapport (le ciel, un bord de l'image) : on ajouterait alors des exemples plus variés, de l'augmentation de données, ou un meilleur modèle.
- Limites : les blocs de SHAP sont grossiers, et six images ne suffisent pas pour conclure ; on en examine davantage avant de ré-annoter.

## À retenir

- L'analyse d'erreurs commence par la matrice de confusion : les paires de classes les plus confondues, calculées et non supposées, disent quelles erreurs regarder d'abord.
- SHAP explique la prédiction d'une image par zones : il masque des blocs de l'image et mesure l'effet sur la probabilité de chaque classe. Ses blocs sont grossiers, et il explique le modèle, pas la réalité.
- Avant de changer de modèle, regarder les étiquettes : des images semblables étiquetées différemment se corrigent dans le guide d'annotation et par ré-annotation, pas avec un meilleur réseau.